# Task 6 (Advanced) - Data Science Business Insights Dashboard
- **Description:** End-to-end data science project that combines analytics, machine learning, and business insights.


## Step 1: Performing complete data analysis

In [4]:
pip install plotly

Note: you may need to restart the kernel to use updated packages.


In [6]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, mean_squared_error

# Cleaned dataset from Task 1
data = pd.read_csv("Cleaned_Netflix_Dataset.csv")
print("Shape:", data.shape)
data.head()

Shape: (8790, 18)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,year_added,month_added,years_to_netflix,duration_value,primary_country,country_count,genre_count,audience
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries,2021,9,1,90,United States,1,1,Teens
1,s3,TV Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",2021,9,0,1,France,1,3,Adults
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",2021,9,0,1,United States,1,3,Adults
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies",2021,9,0,91,Brazil,1,2,Older Kids
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies",2021,9,28,125,United States,1,3,Adults


In [7]:
print("Missing values:", data.isna().sum().sum(), "| Duplicates:", data.duplicated().sum())
print("Release years:", data["release_year"].min(), "-", data["release_year"].max())
data.describe()

Missing values: 0 | Duplicates: 0
Release years: 1925 - 2021


,release_year,year_added,month_added,years_to_netflix,duration_value,country_count,genre_count
count,8790.000000,8790.000000,8790.000000,8790.000000,8790.000000,8790.0,8790.000000
mean,2014.183163,2018.873606,6.655859,4.692378,69.934471,1.0,2.194994
std,8.825466,1.573568,3.436103,8.788835,50.794433,0.0,0.784114
min,1925.000000,2008.000000,1.000000,0.000000,1.000000,1.0,1.000000
25%,2013.000000,2018.000000,4.000000,0.000000,2.000000,1.0,2.000000
50%,2017.000000,2019.000000,7.000000,1.000000,88.500000,1.0,2.000000
75%,2019.000000,2020.000000,10.000000,5.000000,106.000000,1.0,3.000000
max,2021.000000,2021.000000,12.000000,93.000000,312.000000,1.0,3.000000


In [8]:
type_counts = data["type"].value_counts()
top_countries = data.loc[data["country"] != "Unknown", "country"].value_counts().head(10)
top_genres = data["listed_in"].str.split(",").explode().str.strip().value_counts().head(10)

print(pd.DataFrame({"count": type_counts, "percent": (type_counts / len(data) * 100).round(1)}), "\n")
print("Top countries:\n", top_countries, "\n")
print("Top categories:\n", top_genres)

         count  percent
type                   
Movie     6126     69.7
TV Show   2664     30.3 

Top countries:
 country
United States     3240
India             1057
United Kingdom     638
Pakistan           421
Canada             271
Japan              259
South Korea        214
France             213
Spain              182
Mexico             138
Name: count, dtype: int64 

Top categories:
 listed_in
International Movies        2752
Dramas                      2426
Comedies                    1674
International TV Shows      1349
Documentaries                869
Action & Adventure           859
TV Dramas                    762
Independent Movies           756
Children & Family Movies     641
Romantic Movies              616
Name: count, dtype: int64


In [9]:
# Share of TV Shows among titles added each year
added = data[data["year_added"] >= 2014].groupby(["year_added", "type"]).size().unstack(fill_value=0)
added["TV share %"] = (added["TV Show"] / added.sum(axis=1) * 100).round(1)
added

type,Movie,TV Show,TV share %
year_added,,,
2014,19,5,20.8
2015,56,26,31.7
2016,251,175,41.1
2017,836,349,29.5
2018,1237,411,24.9
2019,1424,592,29.4
2020,1284,595,31.7
2021,993,505,33.7


## Step 2: Developing predictive models.

### Model A: Classification (Movie vs TV Show)

In [12]:
y = (data["type"] == "TV Show").astype(int)
data["known_director"] = (data["director"] != "Unknown").astype(int)

categorical = ["rating", "primary_country"]
numeric = ["release_year", "year_added", "month_added", "years_to_netflix",
           "genre_count", "country_count", "known_director"]
# duration and category columns are excluded because they give away the answer

X = data[categorical + numeric]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), categorical),
    ("num", StandardScaler(), numeric),
])
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
}

clf_results = []
for name, model in classifiers.items():
    pipe = make_pipeline(preprocess, model).fit(X_train, y_train)
    pred = pipe.predict(X_test)
    clf_results.append({"Model": name,
                        "Accuracy": round(accuracy_score(y_test, pred), 3),
                        "F1-score": round(f1_score(y_test, pred), 3)})

clf_results = pd.DataFrame(clf_results).sort_values("Accuracy", ascending=False).reset_index(drop=True)
baseline = round(1 - y_test.mean(), 3)
print("Baseline (always Movie):", baseline)
clf_results

Baseline (always Movie): 0.697


,Model,Accuracy,F1-score
0,Random Forest,0.952,0.919
1,Logistic Regression,0.948,0.913
2,Decision Tree,0.944,0.906


### Model B: Forecasting (titles per release year)

In [13]:
series = data["release_year"].value_counts().sort_index().loc[2005:2020]   # 2021 is a partial year
years, counts = series.index.values, series.values

def holt_forecast(train, steps, alpha=0.9, beta=0.2, phi=0.8):
    level, trend = train[0], train[1] - train[0]
    for v in train[1:]:
        new_level = alpha * v + (1 - alpha) * (level + phi * trend)
        trend = beta * (new_level - level) + (1 - beta) * phi * trend
        level = new_level
    return np.array([level + sum(phi ** i for i in range(1, k + 1)) * trend for k in range(1, steps + 1)])

def regression_forecast(train_years, train_counts, future_years, degree):
    m = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    m.fit((train_years - 2005).reshape(-1, 1), train_counts)
    return m.predict((future_years - 2005).reshape(-1, 1))

# Train on 2005-2016, test on 2017-2020
tr = years <= 2016
test_y, test_c = years[~tr], counts[~tr]
preds = {
    "Linear Regression": regression_forecast(years[tr], counts[tr], test_y, 1),
    "Polynomial Regression": regression_forecast(years[tr], counts[tr], test_y, 2),
    "Holt (damped trend)": holt_forecast(counts[tr], len(test_y)),
}
fc_results = pd.DataFrame([
    {"Model": n, "MAE": round(mean_absolute_error(test_c, p), 1),
     "RMSE": round(float(np.sqrt(mean_squared_error(test_c, p))), 1)} for n, p in preds.items()
]).sort_values("RMSE").reset_index(drop=True)
fc_results

,Model,MAE,RMSE
0,Holt (damped trend),100.7,110.7
1,Linear Regression,320.4,340.9
2,Polynomial Regression,260.3,347.0


In [14]:
# Refit the best model on all years and forecast the next 5 years
future_years = np.arange(2021, 2026)
future = np.maximum(holt_forecast(counts, 5), 0).round().astype(int)
forecast_df = pd.DataFrame({"Year": future_years, "Forecast": future})
forecast_df

,Year,Forecast
0,2021,970
1,2022,974
2,2023,978
3,2024,980
4,2025,982


## Step 3: Creating interactive visualizations

Hover over the charts to see values, click legend items to hide or show them, and use the toolbar to zoom.

In [15]:
fig = px.pie(values=type_counts.values, names=type_counts.index,
             title="Content Distribution: Movies vs TV Shows",
             color_discrete_sequence=["#e50914", "#221f1f"])
fig.show()

In [16]:
fig = px.bar(x=top_countries.values, y=top_countries.index, orientation="h",
             title="Top 10 Countries by Number of Titles",
             labels={"x": "Number of Titles", "y": "Country"})
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

In [17]:
fig = px.bar(x=top_genres.values, y=top_genres.index, orientation="h",
             title="Top 10 Categories",
             labels={"x": "Number of Titles", "y": "Category"})
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

In [18]:
trend_df = (data[data["year_added"] >= 2014].groupby(["year_added", "type"]).size()
            .reset_index(name="titles"))
fig = px.line(trend_df, x="year_added", y="titles", color="type", markers=True,
              title="Titles Added to Netflix per Year",
              labels={"year_added": "Year Added", "titles": "Number of Titles"})
fig.show()

In [19]:
# Actual vs forecast of titles per release year
plot_df = pd.concat([
    pd.DataFrame({"Year": years, "Titles": counts, "Series": "Actual"}),
    pd.DataFrame({"Year": future_years, "Titles": future, "Series": "Forecast"}),
])
fig = px.line(plot_df, x="Year", y="Titles", color="Series", markers=True,
              title="Forecast of Titles per Release Year (Holt damped trend)")
fig.show()

In [20]:
fig = px.bar(clf_results, x="Model", y="Accuracy", text="Accuracy",
             title=f"Classifier Accuracy (baseline = {baseline})")
fig.update_yaxes(range=[0.6, 1.0])
fig.add_hline(y=baseline, line_dash="dash")
fig.show()

## Step 4: Generating business recommendations

In [21]:
kids_share = (data["audience"] == "Kids").mean() * 100
adult_share = (data["audience"] == "Adults").mean() * 100
unknown_director = (data["director"] == "Unknown").mean() * 100
one_season = (data.loc[data["type"] == "TV Show", "duration_value"] == 1).mean() * 100

print(f"TV share of additions: {added.loc[2018, 'TV share %']}% (2018) -> {added.loc[2021, 'TV share %']}% (2021)")
print(f"TV shows with only 1 season: {one_season:.0f}%")
print(f"Titles from the top 3 non-US countries: {', '.join(top_countries.index[1:4])}")
print(f"Forecast titles per release year (2021-2025): {future.min()} - {future.max()}")
print(f"Kids-rated titles: {kids_share:.0f}% | Adult-rated titles: {adult_share:.0f}%")
print(f"Titles with no director listed: {unknown_director:.0f}%")

TV share of additions: 24.9% (2018) -> 33.7% (2021)
TV shows with only 1 season: 67%
Titles from the top 3 non-US countries: India, United Kingdom, Pakistan
Forecast titles per release year (2021-2025): 970 - 982
Kids-rated titles: 10% | Adult-rated titles: 46%
Titles with no director listed: 29%


### Recommendations
1. **Invest more in series.** TV shows grew from about 25% of additions in 2018 to 34% in 2021, but 67% of shows have only one season. Renewing proven shows can build lasting audiences.
2. **Grow regional content.** India, the UK and Pakistan follow the US, and International Movies is the top category. Pakistan is TV-led (350 of 421 titles), which is a niche to expand.
3. **Focus on quality over volume.** Titles per release year peaked in 2018 and the forecast is flat (about 970-980 per year), so growth should come from better selection.
4. **Look at family content.** Only about 10% of titles are rated for kids, against 46% for adults. Check this gap with viewing data before investing.
5. **Use "more like this" recommendations** (Task 3) to improve discovery.
6. **Fix metadata gaps.** 29% of titles have no director listed, which weakens search, recommendations and models.

*The dataset has no viewership or revenue data, so these suggestions should be tested against business data.*

## Step 5: Present findings in a professional report

In [22]:
# Summary table of key results (used in the report and dashboard)
report = pd.DataFrame({
    "Metric": ["Total titles", "Movies (%)", "TV Shows (%)", "Top country", "Top category",
               "Best classifier", "Classifier accuracy", "Forecast (titles per year, 2021-2025)"],
    "Value": [len(data), round(type_counts["Movie"] / len(data) * 100, 1),
              round(type_counts["TV Show"] / len(data) * 100, 1),
              top_countries.index[0], top_genres.index[0],
              clf_results.loc[0, "Model"], f"{clf_results.loc[0, 'Accuracy'] * 100:.1f}%",
              f"{future.min()} - {future.max()}"],
})
report.to_csv("Task6_Key_Results.csv", index=False)
report

,Metric,Value
0,Total titles,8790
1,Movies (%),69.7
2,TV Shows (%),30.3
3,Top country,United States
4,Top category,International Movies
5,Best classifier,Random Forest
6,Classifier accuracy,95.2%
7,"Forecast (titles per year, 2021-2025)",970 - 982


### Conclusion
- The catalogue is movie-heavy (69.7%), led by the United States, and mostly aimed at teen and adult viewers.
- TV content is growing in share, while the number of titles per release year peaked in 2018 and has levelled off.
- Movies and TV shows can be told apart with about **95% accuracy** (Random Forest), against a 69.7% baseline.
- The main opportunities are series, regional content and better metadata.

The full written report and the interactive dashboard are available separately.